### This Jupyter notebook demonstrates how to compute pore-level properties of metal-organic frameworks (MOFs) with mofography. Using P1-IRMOF-1 as an example, it builds a periodic pore graph and computes the pore diameter, accessible surface area, accessible volume, and pore size distribution (PSD).

Requirements

- python==3.11.9
- mofography== 2026.10.9.1
- numpy==1.23.5
- pandas==2.2.2
- ase==3.22.1
- networkx==3.2.1

In [1]:
import os

import pandas as pd
import mofography as mgr
from ase.io import read
import numpy as np
import plotly.graph_objects as go

**Distance grid.** Each grid point (0.5 Å spacing) stores its distance to the nearest framework atom surface.

In [2]:
atoms = read('../mofography/examples/TestCIFs/P1-IRMOF-1.cif')
print(f"IRMOF-1: {len(atoms)} atoms, cell = {atoms.cell.cellpar()[:3].round(2)} Å")

spacing = 0.5                                  # Å — see dgrid_methods.qmd
dgrid = mgr.dgrid_from_atoms_rust(atoms, spacing=spacing)
dgrid = dgrid.astype(np.float32)               # cast to float32 — required by do_periodic_segmentation_rust
print(f"dgrid: shape={dgrid.shape}, void max = {dgrid.max():.1f} Å")

IRMOF-1: 424 atoms, cell = [25.83 25.83 25.83] Å
dgrid: shape=(52, 52, 52), void max = 7.5 Å


/projects/academic/kaihangs/czheng37/anaconda3/envs/prf_new/lib/python3.11/site-packages/ase/io/cif.py:401: UserWarning: crystal system 'triclinic' is not interpreted for space group Spacegroup(1, setting=1). This may result in wrong setting!
  warnings.warn(
[RAYON] current_num_threads = 16


**Pore segmentation.** The pore space is split into individual pores by a watershed on the distance grid; periodic images of the same pore are merged.

In [3]:
seg = mgr.do_periodic_segmentation_rust(
    dgrid, atoms, spacing=spacing,
    mask_thickness=0.05, h=0.2, min_distance=1.5, threshold_abs=1.5,
    minimum_window_separation=1.5, wall_thickness=0.05, extra_padding=5,
    compute_connections=True, watershed_backend="rust_det", disable_tqdm=True,
)

[RAYON] current_num_threads = 16
[pocket_maxima_check] Region 1: pocket_radius from maxima_info (5.493 Å) underestimates true region max (5.846 Å) by 0.353 Å; this suggests a missing or under-detected maximum.
[pocket_maxima_check] Region 3: pocket_radius from maxima_info (5.493 Å) underestimates true region max (5.846 Å) by 0.353 Å; this suggests a missing or under-detected maximum.
[pocket_maxima_check] Region 4: pocket_radius from maxima_info (5.284 Å) underestimates true region max (5.629 Å) by 0.345 Å; this suggests a missing or under-detected maximum.
[pocket_maxima_check] Region 5: pocket_radius from maxima_info (5.493 Å) underestimates true region max (5.846 Å) by 0.353 Å; this suggests a missing or under-detected maximum.
[pocket_maxima_check] Region 9: pocket_radius from maxima_info (5.493 Å) underestimates true region max (5.846 Å) by 0.353 Å; this suggests a missing or under-detected maximum.
[pocket_maxima_check] Region 10: pocket_radius from maxima_info (5.493 Å) underest

In [4]:
print(f"Pockets (PBC-grouped):  {len(seg['maxima_pbc'])}")

Pockets (PBC-grouped):  8


**Pore diameter.** Diameter of the largest sphere that fits inside each pore.

In [15]:
print("Pore diameter per pocket:\n")
print(f"  {'pocket':>8s}  {'Diameter (Å)':>14s}  bar")

for pid, i in sorted(
    enumerate(seg["maxima_info"][:len(seg["maxima_pbc"]), 1:4]),
    key=lambda item: dgrid[tuple(item[1])],
    reverse=True,
):
    pore_level_radius = dgrid[tuple(i)]
    pore_level_diameter = 2 * pore_level_radius
    bar = "█" * int(pore_level_diameter)
    print(f"  {pid + 1:>8d}  {pore_level_diameter:>14.2f}  {bar}")

Pore diameter per pocket:

    pocket    Diameter (Å)  bar
         8           15.08  ███████████████
         2           14.37  ██████████████
         6           14.37  ██████████████
         7           14.37  ██████████████
         1           10.99  ██████████
         3           10.99  ██████████
         5           10.99  ██████████
         4           10.57  ██████████


**Accessible surface area.** Monte Carlo estimate of the surface area that a helium probe (radius 1.2 Å) can reach through the periodic pore network, split by pore.

In [16]:
probe_radius=1.2 #helium
sa_mc = mgr.compute_surface_area_mc_winding(
    seg, atoms, probe_radius=probe_radius, n_samples=50000, shell_thickness=0.5
)

print("Accessible SA per pocket (sorted, top 10):\n")
print(f"  {'pocket':>8s}  {'SA (m²/g)':>12s}  bar")
for pid, area in sorted(sa_mc['by_pocket'].items(), key=lambda kv: -kv[1])[:10]:
    bar = "█" * int(area / 5)
    print(f"  {int(pid):>8d}  {area:>12.2f}  {bar}")



Accessible SA per pocket (sorted, top 10):

    pocket     SA (m²/g)  bar
         8        675.53  ███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████
         6        649.24  █████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████
         7        627.67  █████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████
         2        614.86  ██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████
         4        339.79  ███████████████████████████████████████████████████████████████████
         5        303.38  ████████████████████████████████████████████████████████████
         3        297.99  ███████████████████████████████████████████████████████████
         1        296.64  ██████████████

**Accessible pore volume.** Pore volume reachable by the same probe per gram of framework, split by pore.

In [17]:
volume = mgr.compute_void_volume_from_results_winding(seg, atoms, probe_radius=probe_radius)
print("Accessible void volume per pocket (sorted, top 10):\n")
print(f"  {'pocket':>8s}  {'V (cm³/g)':>12s}  bar")
for pid, vol in sorted(volume['by_pocket'].items(), key=lambda kv: -kv[1])[:10]:
    bar = "█" * int(vol * 100)
    print(f"  {int(pid):>8d}  {vol:>12.4f}  {bar}")


Accessible void volume per pocket (sorted, top 10):

    pocket     V (cm³/g)  bar
         8        0.1690  ████████████████
         2        0.1533  ███████████████
         6        0.1533  ███████████████
         7        0.1533  ███████████████
         1        0.0770  ███████
         3        0.0770  ███████
         5        0.0770  ███████
         4        0.0671  ██████


**Pore size distribution (PSD).** A PSD starts from **largest-including-sphere diameters** (Gelb & Gubbins): at each sample site in the void, mofography finds the largest sphere that contains the site without overlapping framework atoms. `compute_psds_pocketwise_rust` draws a fixed number of samples **per segmented pocket** (`regions_pbc`), so each pocket is sampled equally well.

The sampling returns **histograms of diameters**. `mgr.psd_gelb_gubbins(counts, binsize)` converts each histogram into the **Gelb–Gubbins PSD**, \(-\mathrm{d}V/\mathrm{d}D\), with the same 0.25 Å bins as PoreBlazer. We then **divide each curve by its own maximum** (arbitrary units).

In [18]:
num_samples_per_region = 5000   # per pocket
max_diameter = 20.0
binsize = 0.25                  # PoreBlazer default bin width (Å)
bins = np.arange(0, max_diameter + binsize / 2, binsize)


def gelb_gubbins(counts):
    """Diameter histogram -> -dV/dD (the Gelb-Gubbins curve), peak-normalized (a.u.)."""
    _, y = mgr.psd_gelb_gubbins(counts, binsize)
    m = y.max()
    return y / m if m > 0 else y

# diameters (Å) that the -dV/dD values belong to (interior bins)
x_gg, _ = mgr.psd_gelb_gubbins(np.ones(len(bins) - 1), binsize)

In [19]:
pocket_psds = mgr.compute_psds_pocketwise_rust(
    dgrid, seg["regions_pbc"], atoms,
    num_samples_per_region=num_samples_per_region,
    bins=bins,
    material="IRMOF-1",
)


Creating region-wise PSD plots...
Unique region labels found: [1 2 3 4 5 6 7 8]
Found 8 unique regions/pockets
Region 1: 5000 samples, mean pore size: 10.65 Å, std: 2.54 Å
Region 2: 5000 samples, mean pore size: 12.78 Å, std: 3.30 Å
Region 3: 5000 samples, mean pore size: 10.59 Å, std: 2.64 Å
Region 4: 5000 samples, mean pore size: 10.20 Å, std: 2.39 Å
Region 5: 5000 samples, mean pore size: 10.62 Å, std: 2.63 Å
Region 6: 5000 samples, mean pore size: 12.78 Å, std: 3.29 Å
Region 7: 5000 samples, mean pore size: 12.79 Å, std: 3.30 Å
Region 8: 5000 samples, mean pore size: 13.48 Å, std: 3.47 Å


In [20]:
fig_pocketwise = go.Figure()

# Pocket-wise PSDs
colors = ['#66c2a5', '#fc8d62', '#8da0cb', '#e78ac3', '#a6d854', '#ffd92f', '#e5c494', '#d62728']
for i, (label, psd_data) in enumerate(pocket_psds.items()):
    fig_pocketwise.add_trace(go.Scatter(
        x=x_gg, y=gelb_gubbins(psd_data["hist"]), mode="lines",
        name=f"Pocket {label}",
        line=dict(width=2, color=colors[i % len(colors)]),
    ))

fig_pocketwise.update_layout(
    title=f"Pocket-wise Pore Size Distributions (IRMOF-1)<br><sub>{len(pocket_psds)} pockets, {num_samples_per_region} samples each</sub>",
    xaxis_title="Pore diameter (Å)",
    yaxis_title="-dV/dD, peak-normalized (a.u.)",
    legend=dict(x=0.8, y=0.99),
    width=1000,
    height=600,
)
fig_pocketwise.show()